# Entity ***Pools***
+ Layer **gold**
+ Priority **1**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Execution

In [0]:
sv_pools_df = spark.read.table("uniswap.silver.pools")

In [0]:
gld_pools_df = spark.sql(f"""
    WITH cte_silver_pools AS (
        SELECT
            pk_pool_id AS pk_pool_contract,
            token0_id,
            token1_id,
            CONCAT(fee_tier_pct,"%") AS fee_tier_pct,
            fees_USD,
            total_value_locked_USD,
            total_value_locked_ETH,
            total_value_locked_token0,
            total_value_locked_token1,
            volume_USD,
            volume_token0,
            volume_token1,
            liquidity,
            tx_count,
            created_at_block_number,
            created_at_timestamp
        FROM {{df}}
    ),
    cte_join_pools_tokens AS (
        SELECT
            pools.*,
            tokens0.symbol AS token0_symbol,
            tokens0.name AS token0_name,
            tokens1.symbol AS token1_symbol,
            tokens1.name AS token1_name
        FROM cte_silver_pools AS pools
        LEFT JOIN uniswap.silver.tokens AS tokens0
            ON pools.token0_id = tokens0.pk_token_id
        LEFT JOIN uniswap.silver.tokens AS tokens1
            ON pools.token1_id = tokens1.pk_token_id
    )
    SELECT
        pools.*,
        CONCAT(pools.token0_symbol, "/", pools.token1_symbol) AS pair_symbol,
        DATEDIFF(current_date(), pools.created_at_timestamp) AS pool_age_days,
        CASE
            WHEN pools.liquidity = 0 THEN TRUE
            ELSE FALSE
        END AS is_empty_pool,
        CASE
            WHEN pools.total_value_locked_USD >= 1000000 THEN "high_tvl"
            WHEN pools.total_value_locked_USD >= 100000 THEN "medium_tvl"
            WHEN pools.total_value_locked_USD >= 1000 THEN "low_tvl"
            ELSE "dust_tvl"
        END AS tvl_tier,
        context.network,
        context.protocol_version,
        CURRENT_TIMESTAMP() AS _load_timestamp_gold
    FROM cte_join_pools_tokens AS pools
    CROSS JOIN uniswap.silver.factory AS context
""", df = sv_pools_df)

### Save & exit

In [0]:
load_result = load_entity(gld_pools_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)